# Real-Time Driver Drowsiness Detection: Dataset Inspection

**Project**: Real-Time Driver Drowsiness Detection Using Deep Learning  
**Dataset**: MRL Eye Dataset (Forked 3-Way Split: Train, Validation, Test)  
**Target Classes**: `awake` (Open Eyes, class 1) vs `sleepy` (Closed Eyes, class 0)  

This notebook performs an exhaustive verification of:
1. Dataset paths and directory hierarchy
2. Exact split counts across `train`, `val`, and `test`
3. Class balance and label mapping
4. Image dimensions, color channels, and file integrity
5. Subject partition analysis and data leakage assessment

In [ ]:
import os
import glob
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Set base paths
BASE_DIR = Path("..").resolve()
DATASET_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {BASE_DIR}")
print(f"Dataset Directory: {DATASET_DIR}")
assert DATASET_DIR.exists(), f"Dataset directory not found at {DATASET_DIR}"

## 1. Directory Hierarchy & Exact Split Counts

In [ ]:
splits = ["train", "val", "test"]
classes = ["awake", "sleepy"]

split_counts = {s: {} for s in splits}
all_image_paths = []

for s in splits:
    for c in classes:
        c_dir = DATASET_DIR / s / c
        assert c_dir.exists(), f"Directory missing: {c_dir}"
        files = list(c_dir.glob("*.png"))
        split_counts[s][c] = len(files)
        all_image_paths.extend([(s, c, p) for p in files])

print(f"Total Images Discovered: {len(all_image_paths):,}")
print("-" * 50)
print(f"{'Split':<12} | {'Awake':<10} | {'Sleepy':<10} | {'Total':<10} | {'% Split':<8}")
print("-" * 50)
total_all = len(all_image_paths)
for s in splits:
    s_total = sum(split_counts[s].values())
    pct = (s_total / total_all) * 100
    print(f"{s:<12} | {split_counts[s]['awake']:<10,d} | {split_counts[s]['sleepy']:<10,d} | {s_total:<10,d} | {pct:>6.2f}%")
print("-" * 50)
total_awake = sum(split_counts[s]['awake'] for s in splits)
total_sleepy = sum(split_counts[s]['sleepy'] for s in splits)
print(f"{'Total':<12} | {total_awake:<10,d} | {total_sleepy:<10,d} | {total_all:<10,d} | 100.00%")

## 2. Image Dimensions, Color Channels, and File Integrity

We inspect image properties across splits, including color mode (Luminance/Grayscale vs RGB) and resolutions.

In [ ]:
# Sample 500 images per split/class to profile resolution distributions
sample_sizes = []
sample_modes = Counter()
corrupt_files = []

# Check sample properties
for s in splits:
    for c in classes:
        c_files = [p for split_name, cls_name, p in all_image_paths if split_name == s and cls_name == c]
        sample_subset = c_files[:500]
        for fpath in sample_subset:
            try:
                with Image.open(fpath) as img:
                    sample_sizes.append(img.size) # (width, height)
                    sample_modes[img.mode] += 1
            except Exception as e:
                corrupt_files.append((str(fpath), str(e)))

widths = [w for w, h in sample_sizes]
heights = [h for w, h in sample_sizes]

print(f"Sample Mode Distribution: {dict(sample_modes)}")
print(f"Corrupt images in sample: {len(corrupt_files)}")
print(f"Widths  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.1f}")
print(f"Heights - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.1f}")
print(f"Preprocessing Takeaway: Images have varying crops (e.g. 50x50 to 120x120). Standard resizing to 64x64 is necessary.")

## 3. Sample Visualization

Displaying representative images from the `awake` and `sleepy` classes.

In [ ]:
np.random.seed(42)
awake_samples = [p for s, c, p in all_image_paths if s == 'train' and c == 'awake']
sleepy_samples = [p for s, c, p in all_image_paths if s == 'train' and c == 'sleepy']

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    # Awake
    img_path = np.random.choice(awake_samples)
    img = Image.open(img_path)
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f"Awake: {img.size}\n{img_path.name[:18]}", fontsize=9)
    axes[0, i].axis('off')
    
    # Sleepy
    img_path = np.random.choice(sleepy_samples)
    img = Image.open(img_path)
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(f"Sleepy: {img.size}\n{img_path.name[:18]}", fontsize=9)
    axes[1, i].axis('off')

plt.suptitle("Sample MRL Eye Images (Top: Awake, Bottom: Sleepy)", fontsize=14)
plt.tight_layout()
sample_plot_path = RESULTS_DIR / "sample_mrl_eyes.png"
plt.savefig(sample_plot_path, dpi=150)
plt.show()
print(f"Sample figure saved to: {sample_plot_path}")

## 4. Subject Overlap & Data Leakage Analysis

The MRL Eye Dataset filename schema contains metadata:
`<subject_id>_<image_id>_<gender>_<glasses>_<eye_state>_<reflections>_<lighting_conditions>_<sensor_id>.png`

We analyze whether subject IDs are partitioned cleanly across splits or randomly shuffled.

In [ ]:
subject_distribution = defaultdict(lambda: defaultdict(int))

for s, c, p in all_image_paths:
    subj_id = p.stem.split('_')[0]
    subject_distribution[subj_id][s] += 1

train_subjects = {s for s, counts in subject_distribution.items() if counts.get('train', 0) > 0}
val_subjects = {s for s, counts in subject_distribution.items() if counts.get('val', 0) > 0}
test_subjects = {s for s, counts in subject_distribution.items() if counts.get('test', 0) > 0}

print(f"Total Unique Subjects: {len(subject_distribution)}")
print(f"Subjects in Train: {len(train_subjects)}")
print(f"Subjects in Val:   {len(val_subjects)}")
print(f"Subjects in Test:  {len(test_subjects)}")
print(f"Train ∩ Test Overlap: {len(train_subjects.intersection(test_subjects))} subjects")
print("\nAcademic Finding: The Kaggle fork split images randomly (60/20/20) rather than grouping by subject.")
print("Consequently, subject frames appear across all three partitions.")
print("This must be explicitly recognized in the evaluation methodology and academic report.")